<a href="https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rimshakalhoro/flyrank-ml-internship-rimsha/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Finding 1: The research suggests that machine learning can help identify useful patterns in search performance data and support content decisions.

Methodology question: The label or outcome must be checked carefully. If the label is based on later observed search performance, the model may learn patterns associated with that outcome, but this does not prove that the model causes the outcome. The validation design can support a predictive or decision-support claim when future information is kept separate from training data.

Finding 2: The research suggests that combining multiple search signals can be more useful than looking at one metric alone.

Methodology question: The claim depends on how the outcome label was created and whether the validation data was truly independent. A random split can sometimes make performance look stronger when related observations appear in both training and test data. A grouped or time-aware split is a more honest check when the data has groups or time structure.

Overall, I interpret these findings as observed and directional evidence. They support decision-making, but they should not be used to claim that the model proves causation or predicts Google's algorithm.

In [9]:
# Simple methodology check notes

checks = {
    "Finding 1": "Check whether the outcome/label comes from future observed performance.",
    "Finding 2": "Check whether validation keeps related groups or time periods separated.",
    "Main audit rule": "Do not claim causation when the evidence is observational."
}

for finding, check in checks.items():
    print(f"{finding}: {check}")


Finding 1: Check whether the outcome/label comes from future observed performance.
Finding 2: Check whether validation keeps related groups or time periods separated.
Main audit rule: Do not claim causation when the evidence is observational.


I compared the same model under two validation designs.

The first result uses a standard random train/test split. The second result uses a grouped split so that pages from the same content group are kept together.

The grouped result is the more honest estimate when related observations may share similar characteristics. If the grouped score is lower than the random-split score, I should not treat the random-split result as the model's expected real-world performance.

Both numbers are measured estimates from this dataset and validation design. They are directional and should be used for decision-support rather than as proof of future outcomes.

In [10]:
# ============================================================
# ML-09 SECTION 2 — HONEST VALIDATION (BEFORE / AFTER)
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score


# ============================================================
# 1. LOAD DATA
# ============================================================

url = "https://raw.githubusercontent.com/Rimshakalhoro/flyrank-ml-internship-rimsha/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))


# ============================================================
# 2. CREATE TARGET
# ============================================================
# The dataset contains 'ctr', not 'ctr_90d'.

# Only pages with impressions are considered
eligible = df["impressions_90d"].fillna(0) > 0

# Calculate the median CTR among eligible pages
median_ctr = df.loc[eligible, "ctr"].median()

# Proxy target:
# 1 = page has impressions and CTR below the median
# 0 = otherwise

df["needs_attention"] = (
    eligible &
    (df["ctr"] < median_ctr)
).astype(int)

print("\nMedian CTR:", round(median_ctr, 6))

print("\nTarget distribution:")
print(df["needs_attention"].value_counts())

print("\nTarget percentage:")
print(
    (df["needs_attention"]
     .value_counts(normalize=True) * 100)
    .round(2)
)


# ============================================================
# 3. SELECT FINAL FEATURES
# ============================================================
# IMPORTANT:
# We do NOT include:
# - ctr (because it creates the target)
# - clicks_90d (directly related to CTR)
#
# This avoids obvious target leakage.

features = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "trend_direction",
    "trend_pct",
    "impressions_90d"
]

# Keep only columns that exist
features = [
    col for col in features
    if col in df.columns
]

X = df[features].copy()
y = df["needs_attention"].copy()

print("\nFinal features:")
for feature in features:
    print("-", feature)


# ============================================================
# 4. PREPROCESSING
# ============================================================

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# ============================================================
# 5. RANDOM SPLIT — BEFORE
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=8,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)
random_prob = random_model.predict_proba(X_test)[:, 1]

random_accuracy = accuracy_score(
    y_test,
    random_pred
)

random_auc = roc_auc_score(
    y_test,
    random_prob
)


print("\n" + "=" * 55)
print("RANDOM SPLIT RESULTS (BEFORE)")
print("=" * 55)

print(f"Accuracy: {random_accuracy:.4f}")
print(f"ROC-AUC:  {random_auc:.4f}")


# ============================================================
# 6. GROUPED SPLIT — AFTER
# ============================================================
# We use client_id as the grouping variable.
#
# This is more honest because rows from the same client
# are kept together instead of appearing in both train
# and test sets.

groups = df["client_id"].astype(str)

print("\nGrouped validation using: client_id")
print("Number of client groups:", groups.nunique())


gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]


# Create a completely fresh model

group_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=8,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

group_model.fit(
    X_train_group,
    y_train_group
)

group_pred = group_model.predict(
    X_test_group
)

group_prob = group_model.predict_proba(
    X_test_group
)[:, 1]


group_accuracy = accuracy_score(
    y_test_group,
    group_pred
)

group_auc = roc_auc_score(
    y_test_group,
    group_prob
)


# ============================================================
# 7. BEFORE / AFTER COMPARISON
# ============================================================

print("\n" + "=" * 55)
print("GROUPED SPLIT RESULTS (AFTER)")
print("=" * 55)

print(f"Accuracy: {group_accuracy:.4f}")
print(f"ROC-AUC:  {group_auc:.4f}")


comparison = pd.DataFrame({
    "Validation design": [
        "Random split",
        "Grouped split by client_id"
    ],
    "Accuracy": [
        random_accuracy,
        group_accuracy
    ],
    "ROC-AUC": [
        random_auc,
        group_auc
    ]
})

print("\n" + "=" * 55)
print("FINAL BEFORE / AFTER COMPARISON")
print("=" * 55)

display(comparison)


# Save results for the notebook
validation_results = comparison.copy()

Dataset loaded successfully!
Rows: 30000
Columns: 44

Median CTR: 0.07

Target distribution:
needs_attention
0    15190
1    14810
Name: count, dtype: int64

Target percentage:
needs_attention
0    50.63
1    49.37
Name: proportion, dtype: float64

Final features:
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- content_age_days
- days_since_last_update
- avg_position
- trend_direction
- trend_pct
- impressions_90d

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'content_age_days', 'days_since_last_update', 'avg_position', 'trend_pct', 'impressions_90d']

Categorical features:
['competition_level', 'content_type', 'main_intent', 'trend_direction']

RANDOM SPLIT RESULTS (BEFORE)
Accuracy: 0.8043
ROC-AUC:  0.8860

Grouped validation using: client_id
Number of client groups: 32

GROUPED SPLIT RESULTS (AFTER)
Accuracy: 0.8001
ROC-AUC:  0.8799

FINAL BEFORE / AFTER COMPARISON


,Validation design,Accuracy,ROC-AUC
0,Random split,0.804333,0.885952
1,Grouped split by client_id,0.800097,0.879902


I audited the final feature set for possible leakage.

I checked whether any feature directly contains the target, is calculated using future information, or is an outcome that would only be known after the decision point.

The final model features are checked against the target definition. A feature is considered risky if it directly reproduces the label or uses information that would not be available when making the decision.

The audit is important because a high validation score can be misleading when leakage is present. My results should therefore be interpreted as measured only under the information available in the feature design.

In [11]:
# --------------------------------------------------
# LEAKAGE AUDIT
# --------------------------------------------------

target_column = "needs_attention"

final_features = features.copy()

print("TARGET:")
print(target_column)

print("\nFINAL FEATURES:")
for feature in final_features:
    print("-", feature)

# --------------------------------------------------
# RULE 1: TARGET DIRECTLY INSIDE FEATURES
# --------------------------------------------------

direct_target_leakage = target_column in final_features

print("\n" + "=" * 50)
print("CHECK 1: DIRECT TARGET LEAKAGE")
print("=" * 50)

if direct_target_leakage:
    print("WARNING: Target column is included as a feature.")
else:
    print("PASS: Target column is not included as a feature.")

# --------------------------------------------------
# RULE 2: NAME-BASED RISK CHECK
# --------------------------------------------------

risk_words = [
    "target",
    "label",
    "future",
    "outcome",
    "after",
    "post"
]

risky_name_features = [
    feature
    for feature in final_features
    if any(word in feature.lower() for word in risk_words)
]

print("\n" + "=" * 50)
print("CHECK 2: SUSPICIOUS FEATURE NAMES")
print("=" * 50)

if len(risky_name_features) == 0:
    print("PASS: No obvious future/target words found in feature names.")
else:
    print("REVIEW THESE FEATURES:")
    for feature in risky_name_features:
        print("-", feature)

# --------------------------------------------------
# RULE 3: CORRELATION CHECK
# --------------------------------------------------
# This does not prove leakage, but very strong
# correlation can indicate a feature worth reviewing.

numeric_features_for_audit = [
    f for f in final_features
    if pd.api.types.is_numeric_dtype(df[f])
]

print("\n" + "=" * 50)
print("CHECK 3: NUMERIC ASSOCIATION WITH TARGET")
print("=" * 50)

if len(numeric_features_for_audit) > 0:

    audit_frame = df[
        numeric_features_for_audit + [target_column]
    ].copy()

    correlations = (
        audit_frame
        .corr(numeric_only=True)[target_column]
        .drop(target_column)
        .abs()
        .sort_values(ascending=False)
    )

    print(correlations)

else:
    print("No numeric features available for correlation check.")

# --------------------------------------------------
# RULE 4: MANUAL AVAILABILITY CHECK
# --------------------------------------------------

availability_audit = pd.DataFrame({
    "Feature": final_features,
    "Available at decision time?": [
        "Review manually" for _ in final_features
    ],
    "Leakage concern": [
        "Check whether this information exists before the decision"
        for _ in final_features
    ]
})

print("\n" + "=" * 50)
print("MANUAL LEAKAGE AUDIT")
print("=" * 50)

display(availability_audit)

print("""
AUDIT CONCLUSION:
A feature should only be used if it is available at the time
the model or decision-support system would actually make its recommendation.

High correlation alone does not prove leakage.
The key question is whether the feature contains future or outcome information.
""")


TARGET:
needs_attention

FINAL FEATURES:
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- content_age_days
- days_since_last_update
- avg_position
- trend_direction
- trend_pct
- impressions_90d

CHECK 1: DIRECT TARGET LEAKAGE
PASS: Target column is not included as a feature.

CHECK 2: SUSPICIOUS FEATURE NAMES
PASS: No obvious future/target words found in feature names.

CHECK 3: NUMERIC ASSOCIATION WITH TARGET
impressions_90d           0.206475
avg_position              0.189539
word_count                0.112531
days_since_last_update    0.081143
competition               0.051717
search_volume             0.039708
cpc                       0.038703
content_age_days          0.024883
trend_pct                 0.003045
Name: needs_attention, dtype: float64

MANUAL LEAKAGE AUDIT


,Feature,Available at decision time?,Leakage concern
0,search_volume,Review manually,Check whether this information exists before t...
1,competition,Review manually,Check whether this information exists before t...
2,competition_level,Review manually,Check whether this information exists before t...
3,cpc,Review manually,Check whether this information exists before t...
4,content_type,Review manually,Check whether this information exists before t...
5,main_intent,Review manually,Check whether this information exists before t...
6,word_count,Review manually,Check whether this information exists before t...
7,content_age_days,Review manually,Check whether this information exists before t...
8,days_since_last_update,Review manually,Check whether this information exists before t...
9,avg_position,Review manually,Check whether this information exists before t...



AUDIT CONCLUSION:
A feature should only be used if it is available at the time
the model or decision-support system would actually make its recommendation.

High correlation alone does not prove leakage.
The key question is whether the feature contains future or outcome information.



Bold claim:

"My model predicts which pages will perform poorly and tells the team exactly which pages to refresh."

Safer claim:

"In this dataset, the model measured patterns associated with the proxy outcome and produced a directional score that can support decisions about which pages may deserve further review. The result is decision-support evidence, not proof that a page will perform poorly or that refreshing it will cause better search performance."

In [12]:
bold_claim = (
    "My model predicts which pages will perform poorly and tells the "
    "team exactly which pages to refresh."
)

safe_claim = (
    "In this dataset, the model measured patterns associated with the "
    "proxy outcome and produced a directional score that can support "
    "decisions about which pages may deserve further review. The result "
    "is decision-support evidence, not proof of future performance or causation."
)

print("BOLD CLAIM:")
print(bold_claim)

print("\n" + "=" * 50)

print("SAFE CLAIM:")
print(safe_claim)

# Check for careful research language
careful_words = [
    "measured",
    "associated",
    "directional",
    "support",
    "decision-support"
]

print("\n" + "=" * 50)
print("CLAIM LANGUAGE CHECK")
print("=" * 50)

for word in careful_words:
    found = word.lower() in safe_claim.lower()
    print(f"{word}: {'YES' if found else 'NO'}")


BOLD CLAIM:
My model predicts which pages will perform poorly and tells the team exactly which pages to refresh.

SAFE CLAIM:
In this dataset, the model measured patterns associated with the proxy outcome and produced a directional score that can support decisions about which pages may deserve further review. The result is decision-support evidence, not proof of future performance or causation.

CLAIM LANGUAGE CHECK
measured: YES
associated: YES
directional: YES
support: YES
decision-support: YES


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.